In [ ]:
!pip install geopandas metpy xarray[complete] -q
!pip install eccodes cfgrib rioxarray ecmwf.opendata 

In [1]:
import xarray as xr
import cfgrib
import os 
from ecmwf.opendata import Client
# plt.style.available
#import colormaps as cmaps 
import concurrent
import numpy as np


from IPython.display import HTML

# Disable warnings for data download via API
import urllib3
urllib3.disable_warnings()
client = Client(
    source="ecmwf",
    model="ifs",
    resol="0p1",
    preserve_request_order=False,
    infer_stream_keyword=True,
)
import metpy.calc as mpcalc
from metpy.units import units
import json
import pandas as pd
from datetime import datetime, timedelta

import rioxarray
import xarray as xr
import cfgrib

import json
import zipfile
import os
from os import walk
import netCDF4 as nc
import geopandas as gpd
from shapely.geometry import Point
import pandas as pd
import shapely
from shapely.geometry import Point
import xarray as xr
import logging

In [2]:
def dict_l(data, fm, max_mb=15.0):
    """Save a DataArray as compact JSON, split into parts of at most max_mb MB.

    Mirrors the UKMO save_json.py logic: the leading "step" axis is sliced and
    whole slices are packed into parts. One part is written as <fm>.json, more
    than one as <fm>_partNNofNN.json. A small <fm>_manifest.json lists them.
    """
    try:
        data = data.squeeze()
        dims = list(data.dims)
        arr = np.asarray(data.values)
        tdim = dims.index("step") if ("step" in dims and arr.ndim > 1) else None
        if tdim is not None and tdim != 0:
            arr = np.transpose(arr, [tdim] + [i for i in range(arr.ndim) if i != tdim])
            dims = [dims[tdim]] + [d for i, d in enumerate(dims) if i != tdim]

        def round_numbers(obj):
            if isinstance(obj, dict):
                return {k: round_numbers(v) for k, v in obj.items()}
            elif isinstance(obj, (list, tuple)):
                return [round_numbers(item) for item in obj]
            elif isinstance(obj, (float, np.floating)):
                val = float(obj)
                if val != val or val in (float("inf"), float("-inf")):
                    return None
                return round(val, 2)
            elif isinstance(obj, (int, np.integer)):
                return int(obj)
            else:
                return obj

        def convert_types(obj):
            if isinstance(obj, dict):
                return {k: convert_types(v) for k, v in obj.items()}
            elif isinstance(obj, (list, tuple)):
                return [convert_types(item) for item in obj]
            elif isinstance(obj, str):
                return str(obj)
            elif obj is None:
                return None
            elif isinstance(obj, (np.datetime64, pd.Timestamp, datetime)):
                return obj.isoformat()
            elif isinstance(obj, timedelta):
                return obj.total_seconds()
            elif isinstance(obj, np.ndarray):
                return obj.tolist()
            elif isinstance(obj, (bool, np.bool_)):
                return bool(obj)
            elif isinstance(obj, (np.floating, float)):
                return float(obj)
            elif isinstance(obj, (np.integer, int)):
                return int(obj)
            raise TypeError(f"Type {type(obj)} not serializable")

        raw = data.to_dict(data="list", encoding=False)
        coords = round_numbers(convert_types(raw.get("coords", {})))
        attrs = round_numbers(convert_types(raw.get("attrs", {})))

        def dumps(obj):
            return json.dumps(obj, separators=(",", ":"), default=convert_types)

        def clean(a):
            """NumPy array -> nested lists, floats rounded to 2 dp, NaN/Inf -> None."""
            a = np.asarray(a)
            if a.dtype.kind == "f":
                a = np.round(a.astype(np.float64), 2)
                mask = np.isfinite(a)
                if not mask.all():
                    a = a.astype(object)
                    a[~mask] = None
            return a.tolist()

        if tdim is not None:
            slices = [dumps(clean(arr[i])) for i in range(arr.shape[0])]
        else:
            slices = [dumps(clean(arr))]

        def part_coords(a, b):
            out = {}
            for k, v in coords.items():
                if (isinstance(v, dict) and v.get("dims") and v["dims"][0] == "step"
                        and isinstance(v.get("data"), list) and len(v["data"]) >= b):
                    out[k] = dict(v, data=v["data"][a:b])
                else:
                    out[k] = v
            return out

        head = '{"dims":' + dumps(dims) + ',"attrs":' + dumps(attrs)
        overhead = len(head) + len(dumps(coords)) + 40
        budget = max(int(max_mb * 1e6) - overhead, 1)
        n = len(slices)
        groups, start = [], 0
        while start < n:
            used, end = 0, start
            while end < n:
                add = len(slices[end]) + 1
                if end > start and used + add > budget:
                    break
                used += add
                end += 1
            groups.append((start, end))
            start = end

        nparts = len(groups)
        old = os.path.join(os.getcwd(), f"{fm}.json")
        if nparts > 1 and os.path.exists(old):
            os.remove(old)
        files = []
        for gi, (a, b) in enumerate(groups, 1):
            name = f"{fm}.json" if nparts == 1 else f"{fm}_part{gi:02d}of{nparts:02d}.json"
            payload = (head + ',"coords":' + dumps(part_coords(a, b))
                       + ',"data":[' + ",".join(slices[a:b]) + "]}")
            logging.info("Saving data to %s ...", name)
            with open(os.path.join(os.getcwd(), name), "w") as json_file:
                json_file.write(payload)
            files.append(name)

        with open(os.path.join(os.getcwd(), f"{fm}_manifest.json"), "w") as json_file:
            json.dump({"fm": fm, "count": nparts, "max_mb": max_mb, "parts": files},
                      json_file, indent=4)
        logging.info("Data successfully saved in %d part(s).", nparts)
        return files
    except Exception as e:
        logging.error(f"An error occurred: {e}")
        raise

In [ ]:
# HRES:

# fc: Forecast.
# ENS:

# cf: Control forecast.
# pf: Perturbed forecast.
# em: Ensemble mean.
# es: Ensemble standard deviation.
# ep: Probabilities.
# Valid values for stream are:

# oper: Atmospheric fields from HRES - 00 UTC and 12 UTC.
# wave: Ocean wave fields from HRES - 00 UTC and 12 UTC.
# enfo: Atmospheric fields from ENS.
# waef: Ocean wave fields from ENS.
# scda: Atmospheric fields from HRES - 06 UTC and 18 UTC.
# scwv: Ocean wave fields from HRES - 06 UTC and 18 UTC.

In [ ]:
step = list(range(3, 145, 3))

client.retrieve(
    time=12,
    type="fc",
    stream='oper',step=step,
    param=['d','gh',"r","vo"],
    levelist=[ 850, 500],
    target="atms.grib2",
)

In [ ]:
ds_850=xr.open_dataset('atms.grib2',engine='cfgrib').sel(longitude=slice(60,100),latitude=slice(40,0),isobaricInhPa=850)
ds_500=xr.open_dataset('atms.grib2',engine='cfgrib').sel(longitude=slice(60,100),latitude=slice(40,0),isobaricInhPa=500)


In [ ]:
gh8=ds_850['gh']
d8=ds_850['d']
r8=ds_850['r']
vo8=ds_850['vo']

dict_l(gh8,"gh850")
dict_l(d8*1e5,"div850")
dict_l(r8,"rh850")
dict_l(vo8*1e5,"vo850")




gh5=ds_500['gh']
d5=ds_500['d']
r5=ds_500['r']
vo5=ds_500['vo']

dict_l(gh5,"gh500")
dict_l(d5*1e5,"div500")
dict_l(r5,"rh500")
dict_l(vo5*1e5,"vo500")

In [ ]:
step = list(range(3, 73, 3))


client.retrieve(
    time=12,
    type="fc",
    stream='oper',step=step,
    param=['tp','tcwv',"msl","2t"],
    target="atms.grib2",
)




In [ ]:
# import cfgrib
t2 = xr.open_dataset('atms.grib2',engine='cfgrib',decode_coords="all",
                     backend_kwargs={ 'filter_by_keys':{ 'cfVarName': 't2m',}}).sel(longitude=slice(60,100),latitude=slice(40,0))['t2m']
ds=xr.open_dataset('atms.grib2',engine='cfgrib')
rf = ds.sel(longitude=slice(60,100),latitude=slice(40,0))['tp'][:, :, :]  # Select a specific time slice
p =ds.sel(longitude=slice(60,100),latitude=slice(40,0))['msl'][:, :, :]/100  # Select a specific time slice
tcw=ds.sel(longitude=slice(60,100),latitude=slice(40,0))['tcwv']

rf2=rf* units('m').to(units.mm)


In [ ]:
dict_l(t2,"2m_temp")
dict_l(rf2,"rf")
dict_l(p,"msl")
dict_l(tcw,"tcwv")
# dict(w10,"w10")
# dict(tcw,"tcwv")
